In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import GPT2Config, GPT2LMHeadModel
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df = pd.read_csv('llmtrace_news.csv')
texts = df['text'].dropna().tolist()
print(f"Текстов: {len(texts)}")

Device: cpu
Текстов: 29426


In [ ]:
with open('corpus.txt', 'w', encoding='utf-8') as f:
    for t in texts:
        f.write(t + '\n')

spm.SentencePieceTrainer.train(
    input='corpus.txt', model_prefix='bpe5k', vocab_size=5000,
    model_type='bpe', character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load('bpe5k.model')
print(f"Vocab BPE 5k: {sp.get_piece_size()}")

token_counts = Counter()
for text in texts:
    token_counts.update(sp.encode(text))

top10 = set([t for t, _ in token_counts.most_common(10)])
unk = sp.unk_id()
print(f"Топ-10 для удаления: {top10}")

raw_tokens = []
for text in texts:
    for t in sp.encode(text):
        raw_tokens.append(unk if t in top10 else t)

unique_tokens = sorted(set(raw_tokens))
old_to_new = {old: new for new, old in enumerate(unique_tokens)}
tokens_list = [old_to_new[t] for t in raw_tokens]
vocab_size = len(unique_tokens)
print(f"Vocab после обрезки и ремаппинга: {vocab_size}")

Vocab BPE 5k: 5000
Топ-10 для удаления: {5, 7, 71, 13, 4301, 4304, 18, 19, 55, 27}
Vocab после обрезки и ремаппинга: 4986


In [ ]:
freqs = np.array(list(token_counts.values()))
freqs_sorted = np.sort(freqs)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(50), freqs_sorted[:50])
plt.title('Топ-50 токенов (после обрезки)')
plt.xlabel('Ранг')
plt.ylabel('Частота')
plt.show()

print(f"Топ-1: {freqs_sorted[0]}, Топ-50: {freqs_sorted[49]}, Мин: {freqs_sorted[-1]}")

In [ ]:
block_size = 64
batch_size = 32

all_tokens = torch.tensor(tokens_list, dtype=torch.long)
n_blocks = len(all_tokens) // (block_size + 1)
X = all_tokens[:n_blocks * (block_size + 1)].view(-1, block_size + 1)
inputs, targets = X[:, :-1], X[:, 1:]
dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"Блоков: {n_blocks}")

Блоков: 203515


In [ ]:
def train(name, loader, vocab_size, lr, epochs=100):
    config = GPT2Config(
        vocab_size=vocab_size, n_positions=64, n_embd=128, n_layer=2, n_head=2,
        bos_token_id=0, eos_token_id=1,
    )
    model = GPT2LMHeadModel(config).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr) if name=='sgd' else torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        model.train()
        total = 0
        batches = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = model(xb, labels=yb).loss
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
            batches += 1
        avg = total / batches
        history.append(avg)
        if (epoch+1) % 20 == 0:
            print(f"{name} epoch {epoch+1}: loss={avg:.4f}")
    return history

print("=== SGD ===")
loss_sgd = train("sgd", loader, vocab_size, lr=0.5)
np.save("bpe5k_top10_sgd.npy", np.array(loss_sgd))

print("\n=== Adam ===")
loss_adam = train("adam", loader, vocab_size, lr=0.001)
np.save("bpe5k_top10_adam.npy", np.array(loss_adam))

plt.plot(loss_sgd, label='SGD')
plt.plot(loss_adam, label='Adam')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.title('BPE 5k + удаление топ-10 токенов')
plt.savefig("bpe5k_top10.png", dpi=150)
plt.show()

print(f"SGD final: {loss_sgd[-1]:.4f}, Adam final: {loss_adam[-1]:.4f}, Delta: {loss_sgd[-1]-loss_adam[-1]:.4f}")